# Simple Multi-Band Batch Injection Demo (g, r, i)

This notebook scales the i-band batch demo across multiple Rubin filters.

1. Set up the project path and imports
2. Configure the run with `bands=['g', 'r', 'i']`
3. Load the **same** `(tract, patch)` in all three bands at once
4. Plug in your detector
5. Call `pipe.run_batch_multiband` — the **same random catalog** is reused
   across bands (so cluster positions are physically consistent), but each
   band uses its own image + PSF for injection and detection
6. Build a per-band `ClusterRetrieval` and produce per-band completeness
   curves plus a side-by-side comparison plot

The structure mirrors `simple_batch_injection_demo.ipynb` — only the
config, load step, run call, and analysis loop change.

In [ ]:
import sys
from pathlib import Path

repo_root = Path('/home/snair63/WORK/INJECT/star-cluster-injection-pipeline').resolve()
sys.path.insert(0, str(repo_root))

from lsst.daf.butler import Butler
from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline
from src.retrieval import ClusterRetrieval
from src.plotting import plot_completeness_1d

## 1. Configure The Run

The only difference from the single-band demo is `bands=['g', 'r', 'i']`
in `InjectionConfig`. When `bands` is set it overrides the single `band`
value, and `pipe.config.active_bands` returns the list.

In [ ]:
BUTLER_REPO = 'dp02'
BUTLER_COLLECTIONS = '2.2i/runs/DP0.2'
DATA_ID = {'tract': 3828, 'patch': 24}   # no 'band' key — multi-band load

BANDS = ['g', 'r', 'i']

config = InjectionConfig(
    run_name='simple_multiband_demo',
    bands=BANDS,
    cutout_size=1200,
    n_clusters=1000,
    seed=42,
    edge_buffer=50,
    use_actual_psf=True,
    psf_fwhm_fallback=3.5,
    output_dir=str(repo_root / 'plots' / 'simple_multiband_demo'),
    cluster_config=ClusterConfig(
        profile_type='king',
        mag_min=20.0,
        mag_max=26.0,
        r_half_min=2.0,
        r_half_max=10.0,
        concentration_min=5.0,
        concentration_max=30.0,
    ),
)

config

## 2. Load All Three Bands

`pipe.load_data(butler=butler, data_id=DATA_ID)` loops over
`config.active_bands` and stores per-band `images`, `psf_objs`, and
`bboxes` dicts on the pipeline. `pipe.image` is set to the first band's
image so any single-band code keeps working.

In [ ]:
butler = Butler(BUTLER_REPO, collections=BUTLER_COLLECTIONS)
pipe = InjectionPipeline(config)
pipe.load_data(butler=butler, data_id=DATA_ID)

for b in BANDS:
    print(f'  band {b}: shape={pipe.images[b].shape}  bbox_origin={pipe.bboxes[b]}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, len(BANDS), figsize=(5 * len(BANDS), 5),
                         constrained_layout=True)
for ax, b in zip(np.atleast_1d(axes), BANDS):
    img = pipe.images[b]
    vmin, vmax = np.percentile(img, [1, 99])
    ax.imshow(img, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
    ax.set_title(f'{b}-band coadd ({img.shape[1]} x {img.shape[0]} px)')
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 3. Detection Function — Plug In Your Own Detector

Same contract as the single-band demos: a callable that takes an image
and returns `list[dict]` with `x` and `y` keys. `run_batch_multiband`
applies the same `detector_fn` to every band's injected image.

The example below is a stand-in calling the matched-filter + MCI
detector with one shared parameter set across bands. **For real science
you usually want band-specific tuning** (different `psf_fwhm`,
`threshold_sigma`, `mci_max`). The simplest way to get that is to skip
`run_batch_multiband` and call `pipe.run_batch` per band yourself with a
different `detector_fn` each time — see the optional cell at the end.

In [ ]:
# ======================================================================
# >>> PLUG IN YOUR DETECTOR HERE <<<
#
# Replace the body of `mci_detector` with your own detection code.
# Contract: take a 2D image array, return list[dict] with at least
# 'x' and 'y' pixel coordinates per detection.
# ======================================================================
from src.detection import run_cluster_detection


def mci_detector(image):
    """Stand-in detector — replace with your own method."""
    return run_cluster_detection(
        image=image,
        psf_fwhm=config.psf_fwhm_fallback,
        threshold_sigma=3.0,
        mci_max=0.85,
        snr_min=3.0,
        r_half_min=1.0,
        ellipticity_max=0.6,
        box_size=64,
        pixel_scale=config.pixel_scale,
        use_multiscale=True,
        use_mci=True,
        verbose=False,
    )

## 4. Run Batch Injection Across All Bands

`pipe.run_batch_multiband` does, per iteration:

1. Generate **one** random truth catalog (same `(x, y, mag, r_half, ...)`
   for every band — physically correct, since a cluster lives at one
   place on the sky)
2. For each band, inject that catalog into a copy of that band's image
   using that band's real Rubin PSF
3. Run `mci_detector` on each band's injected image
4. Checkpoint per band into `<checkpoint_dir>/<band>/`

It returns `dict[band → list[iteration_dict]]` with the same shape as
`run_batch`'s return value, one entry per band.

`psf_fwhm_fallbacks` lets you set per-band fallback FWHMs (used only if
the coadd PSF object is unavailable). The defaults here are rough
i-band-ish placeholders — tune to your data.

In [ ]:
N_ITERATIONS = 10
N_PER_ITER   = 1000

PSF_FWHM_FALLBACKS = {'g': 3.8, 'r': 3.5, 'i': 3.4}

results = pipe.run_batch_multiband(
    n_iterations=N_ITERATIONS,
    n_per_iter=N_PER_ITER,
    psf_fwhm_fallbacks=PSF_FWHM_FALLBACKS,
    detector_fn=mci_detector,
    store_images=False,
    checkpoint_dir=str(repo_root / 'plots' / 'simple_multiband_demo' / 'checkpoints'),
    n_workers=1,
    use_psf_cache=True,
    psf_cache_grid=8,
    psf_cache_size=2000,
    verbose=True,
)

for b in BANDS:
    n_inj = sum(len(it['injection_info']) for it in results[b])
    n_det = sum(len(it['detections'])     for it in results[b])
    print(f'  band {b}: iterations={len(results[b])}  injected={n_inj}  detected={n_det}')

## 5. Per-Band Retrieval And Completeness

`pipe.run_batch_multiband` does **not** populate `pipe.injection_info`
or `pipe.detection_catalog` with the multi-band results — those
attributes only hold one band's worth at a time. So we build a
`ClusterRetrieval` per band by hand from the pooled per-band lists.

In [ ]:
MATCH_RADIUS_PX = 5.0

retrievals = {}
summaries  = {}

for b in BANDS:
    pooled_inj = [e for it in results[b] for e in it['injection_info']]
    pooled_det = [d for it in results[b] for d in it['detections']]

    r = ClusterRetrieval(pooled_inj, pooled_det)
    r.match_detections(match_radius=MATCH_RADIUS_PX)
    retrievals[b] = r
    summaries[b]  = r.get_summary_statistics()

    s = summaries[b]
    print(f'  band {b}: overall={s["overall_completeness"]:.1%}  '
          f'mag_50={s["mag_50_limit"]:.2f}  '
          f'r_half_50={s["r_half_50_limit"]:.2f} px')

## 6. Plot Per-Band Completeness Curves

First, the standard `plot_completeness_1d` once per band (mag and r_half
side by side). Then a single overlay panel comparing the magnitude
completeness curves of all bands on one axes — that's the headline
multi-band plot.

In [ ]:
PLOT_OUTPUT_DIR = Path(config.output_dir) / 'plots'
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for b in BANDS:
    fig, _ = plot_completeness_1d(retrievals[b], config,
                                  pixel_scale=config.pixel_scale)
    fig.suptitle(f'{b}-band completeness', y=1.02)
    save_path = PLOT_OUTPUT_DIR / f'completeness_1d_{b}.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  saved {save_path}')

In [ ]:
# Side-by-side overlay: magnitude completeness for all bands on one axes.
cc = config.cluster_config
mag_bins = np.arange(cc.mag_min, cc.mag_max + 0.5, 0.5)
rh_bins  = np.linspace(cc.r_half_min, cc.r_half_max, 10)

BAND_COLORS = {'u': 'tab:purple', 'g': 'tab:blue', 'r': 'tab:green',
               'i': 'tab:orange', 'z': 'tab:red',  'y': 'tab:brown'}

fig, (ax_mag, ax_rh) = plt.subplots(1, 2, figsize=(13, 5))

for b in BANDS:
    r = retrievals[b]
    color = BAND_COLORS.get(b, None)

    bc, comp, err = r.compute_completeness('magnitude', mag_bins)
    ax_mag.step(bc, comp, where='mid', lw=2, color=color, label=f'{b}')
    ax_mag.fill_between(bc, np.clip(comp - err, 0, 1), np.clip(comp + err, 0, 1),
                        step='mid', alpha=0.15, color=color)

    bc, comp, err = r.compute_completeness('r_half', rh_bins)
    ax_rh.step(bc, comp, where='mid', lw=2, color=color, label=f'{b}')
    ax_rh.fill_between(bc, np.clip(comp - err, 0, 1), np.clip(comp + err, 0, 1),
                       step='mid', alpha=0.15, color=color)

for ax in (ax_mag, ax_rh):
    ax.axhline(0.5, color='gray', ls='--', lw=1)
    ax.axhline(0.9, color='gray', ls=':',  lw=1)
    ax.set_ylim(0, 1.05)
    ax.legend(title='Band', loc='lower left')

ax_mag.set_xlabel('Injected magnitude (AB)')
ax_mag.set_ylabel('Completeness')
ax_mag.set_title('Completeness vs Magnitude (per band)')

ax_rh.set_xlabel('Half-light radius (px)')
ax_rh.set_ylabel('Completeness')
ax_rh.set_title('Completeness vs Half-light radius (per band)')

plt.tight_layout()
save_path = PLOT_OUTPUT_DIR / 'completeness_1d_overlay.png'
fig.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'  saved {save_path}')

## 7. Optional: Band-Specific Detector Tuning

`run_batch_multiband` runs **one** detector function across every band.
If you want band-specific detection knobs (e.g. different `mci_max` or
`psf_fwhm` per band), bypass `run_batch_multiband` and call
`pipe.run_batch` per band yourself, with a closure that captures the
band's settings. Sketch:

```python
def make_detector(psf_fwhm, mci_max):
    def _det(image):
        return run_cluster_detection(
            image=image, psf_fwhm=psf_fwhm, mci_max=mci_max,
            threshold_sigma=3.0, snr_min=3.0,
            pixel_scale=config.pixel_scale,
            use_multiscale=True, use_mci=True, verbose=False,
        )
    return _det

PER_BAND = {
    'g': dict(psf_fwhm=3.8, mci_max=0.90),
    'r': dict(psf_fwhm=3.5, mci_max=0.85),
    'i': dict(psf_fwhm=3.4, mci_max=0.85),
}

results = {}
for b in BANDS:
    pipe.image = pipe.images[b]
    bbx, bby   = pipe.bboxes[b]
    results[b] = pipe.run_batch(
        n_iterations=N_ITERATIONS,
        n_per_iter=N_PER_ITER,
        psf_obj=pipe.psf_objs[b],
        bbox_x_min=bbx, bbox_y_min=bby,
        psf_fwhm_fallback=PER_BAND[b]['psf_fwhm'],
        detector_fn=make_detector(**PER_BAND[b]),
        band=b,
        verbose=True,
    )
```

Everything downstream (per-band retrieval, completeness, overlay) is
identical to sections 5 and 6.